# Preparar y estructurar datos de Instacart con Pandas

**Aprendiz:** Santiago Varela Peña
**Evidencia:** GA1-220501094-01-AA1-EV06 – Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*:
explicación con ejemplos, laboratorio guiado y reto de cada lección. Cada lección usa las variables de la anterior,
por eso se debe ejecutar de arriba abajo (**Entorno de ejecución → Ejecutar todas**).

## Preparación del entorno

Los cinco CSV usan `;` como separador. El cuaderno funciona en dos entornos:

- **Google Colab:** monta Google Drive y lee los archivos de `MyDrive/instacart-datasets/`.
- **Local (Jupyter / VSCode):** lee los archivos de `data/raw/` (ejecutando el cuaderno desde la carpeta `notebooks/`).

In [1]:
from pathlib import Path

import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA = Path('/content/drive/MyDrive/instacart-datasets')
except ModuleNotFoundError:
    RUTA = Path('..') / 'data' / 'raw'

print('Pandas', pd.__version__)
print('Datos en:', RUTA)
print(sorted(p.name for p in RUTA.glob('*.csv')))

Pandas 3.1.1
Datos en: ..\data\raw
['aisles.csv', 'departments.csv', 'instacart_orders.csv', 'order_products.csv', 'products.csv']


> Los resultados esperados del material (por ejemplo, la memoria en MB) se calcularon con **pandas 3**.
> Si la celda anterior muestra una versión 2.x, los conteos de filas y nulos coinciden igual, pero las cifras de memoria
> pueden variar un poco porque pandas 2 guarda el texto como `object`.

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')

for nombre, df in [('orders', orders), ('products', products), ('aisles', aisles),
                   ('departments', departments), ('order_prods', order_prods)]:
    print(f'{nombre:12} {df.shape}')

orders       (478967, 6)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)
order_prods  (4545007, 4)


# Lección 2: Limpieza y estandarización de datos crudos

Resolveremos los problemas encontrados en la exploración: duplicados, tipos de dato incorrectos, nulos, texto sin
estandarizar y duplicados implícitos. Para cada problema hay tres opciones (**eliminar, corregir o conservar**) y la
elección depende de qué significa el dato. Orden: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

### Duplicados exactos: detectar con `duplicated` y eliminar con `drop_duplicates`

Primero se **detectan** con `duplicated(keep=False)`, que marca todas las copias sin borrar nada.

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


*Esperado:* `30` filas (15 pedidos y sus 15 copias), todas con `order_dow = 3` y `order_hour_of_day = 2`
(miércoles a las 2 a. m.). Que ocurran en la misma franja apunta a un fallo del sistema, no a pedidos reales:
es seguro borrarlas.

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


*Esperado:* `(478952, 6) True`.

### Tipos de dato con `astype`

`int8`, `int16` e `int32` reducen memoria según el rango de cada columna. `Int16` (con mayúscula) es *nullable*:
admite enteros y nulos a la vez.

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


*Esperado:* `Antes: 23.0 MB` y `Después: 7.2 MB`.

### Nulos: comprobar la causa antes de decidir

Hipótesis: los nulos de `days_since_prior_order` son los primeros pedidos de cada cliente.

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


*Esperado:* `True`. Cada nulo es un primer pedido: no existe un pedido anterior. Decisión: **conservar** el nulo.
Rellenar con 0 inventaría que el cliente volvió el mismo día y bajaría la media de espera.

In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


*Esperado:* una sola combinación, pasillo `100` y departamento `21` (`missing`), con 1 258 productos, y después `0`
nulos. Decisión: **corregir** con un valor explícito, porque el producto existe y no se puede borrar.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16


0 0


*Esperado:* `Espacios dobles: 403`, `Espacio invisible: 16` y, tras limpiar, `0 0`.
El espacio no separable (`'\xa0'`) se reemplaza en un paso propio porque en pandas 3 la expresión `\s` no lo reconoce.

### Duplicados implícitos

Algunos duplicados solo aparecen después de estandarizar, como `American Cheese slices` y `American Cheese Slices`.

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


*Esperado:* `141` nombres repetidos y `268 de 280` productos implicados que sí aparecen en pedidos.
Decisión: **conservar y documentar**. Tienen `product_id` distintos y borrarlos dejaría pedidos apuntando a productos
inexistentes; en los análisis se agrupará por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


*Esperado:* `(478952, 6)`: aplicada a una tabla ya limpia, la función no cambia nada.

Verificación de toda la limpieza (si la celda no muestra nada, la limpieza está completa):

In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Un **pipeline** es una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola
instrucción y deja un **informe** de lo que cambió en cada paso. Se trabaja sobre copias nuevas de los archivos.

### Paso 1: recargar los datos crudos

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


*Esperado:* `(478967, 6) (49694, 4) (4545007, 4)`.

### Paso 2: una función que anota el estado de la tabla

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

`anotar` devuelve la tabla **sin cambios**: así se puede intercalar entre pasos sin alterar el resultado.

### Paso 3: un paso de limpieza por función

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [15]:
registro = []   # se reinicia para que el informe no tenga filas repetidas si la celda se ejecuta dos veces

orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [16]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods`
  de 145,4 a 54,5 MB.
- Estandarizar nombres dejó `products` sin nulos (los 1 258 nombres vacíos pasaron a `'Unknown'`).
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos.

### Paso 6: validar el resultado

In [17]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


*Esperado:* `Validación superada` y `True True`: el pipeline produce exactamente las mismas tablas que la limpieza
manual de la explicación.

**Conclusiones del laboratorio**

- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una función de registro produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "¿qué le hiciste a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## Reto 2

### Ejercicio 1
Comprobar si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si
`reordered` solo contiene 0 y 1.

In [18]:
dup_exactos = order_prods.duplicated().sum()
dup_pares = order_prods.duplicated(subset=['order_id', 'product_id']).sum()
valores_reordered = sorted(order_prods['reordered'].unique().tolist())

print('Filas duplicadas exactas:', dup_exactos)
print('Pares order_id + product_id repetidos:', dup_pares)
print('Valores de reordered:', valores_reordered)
print('¿reordered solo tiene 0 y 1?', set(valores_reordered) <= {0, 1})

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿reordered solo tiene 0 y 1? True


**Respuesta:** `order_prods` tiene **0 filas duplicadas exactas** y **0 pares `order_id` + `product_id` repetidos**:
cada producto aparece una sola vez por pedido, así que la tabla no necesita deduplicarse. `reordered` solo contiene
**0 y 1**, como corresponde a un indicador binario de recompra.

### Ejercicio 2
Comprobar si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final.
¿Necesitan limpieza de texto?

In [19]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    texto = df[col]
    print(f'{nombre}: {len(df)} filas')
    print('  nombres duplicados:', texto.duplicated().sum())
    print('  duplicados sin distinguir mayúsculas:', texto.str.strip().str.lower().duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())
    print('  nulos:', texto.isna().sum())

aisles: 134 filas
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  nulos: 0
departments: 21 filas
  nombres duplicados: 0
  duplicados sin distinguir mayúsculas: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
  nulos: 0


**Respuesta:** **no necesitan limpieza de texto.** En `aisles` (134 filas) y `departments` (21 filas) todos los conteos
son 0: no hay nombres duplicados (ni siquiera ignorando mayúsculas), ni espacios al inicio o al final, ni espacios
dobles, ni nulos. El único caso especial es el departamento `'missing'` (id 21), que no es un problema de texto sino de
significado; se trata en el ejercicio 7.

### Ejercicio 3
Investigar los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo,
cada uno? ¿Qué los explica?

In [20]:
nulos_carrito = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_afectados = nulos_carrito['order_id'].unique()
detalle = order_prods[order_prods['order_id'].isin(pedidos_afectados)]

productos_por_pedido = detalle.groupby('order_id').size()
posicion_maxima = detalle.groupby('order_id')['add_to_cart_order'].max()

print('Nulos en add_to_cart_order:', len(nulos_carrito))
print('Pedidos en los que ocurren:', len(pedidos_afectados))
print('Productos por pedido afectado: mínimo', productos_por_pedido.min(), '| máximo', productos_por_pedido.max())
print('Posición máxima registrada en esos pedidos:', posicion_maxima.max())
print('Posición máxima registrada en toda la tabla:', order_prods['add_to_cart_order'].max())
print('¿Todos los pedidos afectados tienen más productos que la posición máxima registrada?',
      (productos_por_pedido > order_prods['add_to_cart_order'].max()).all())

Nulos en add_to_cart_order: 836
Pedidos en los que ocurren: 70
Productos por pedido afectado: mínimo 65 | máximo 127
Posición máxima registrada en esos pedidos: 64
Posición máxima registrada en toda la tabla: 64


¿Todos los pedidos afectados tienen más productos que la posición máxima registrada? True


**Respuesta:** los 836 nulos ocurren en solo **70 pedidos**, y cada uno tiene **como mínimo 65 productos** (el más
grande, 127). La posición máxima registrada, tanto en esos pedidos como en toda la tabla, es **64**. La explicación es
que el sistema solo guardó la posición en el carrito hasta 64: a partir del producto 65 la posición quedó vacía.
Por eso se **conservan** los nulos: el producto sí se compró y solo falta su posición; borrar esas filas haría perder
artículos reales de los pedidos más grandes.

### Ejercicio 4
Cargar de nuevo `products.csv` en `products_crudo` y contar cuántos nombres (no nulos) cambiaron con la estandarización.

In [21]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

tenia_nombre = products_crudo['product_name'].notna()
original = products_crudo.loc[tenia_nombre, 'product_name']
estandarizado = products.loc[tenia_nombre, 'product_name']

cambiaron = (original != estandarizado)
print('Nombres no nulos:', tenia_nombre.sum())
print('Nombres que cambiaron con la estandarización:', cambiaron.sum())
print(f'Porcentaje: {cambiaron.mean() * 100:.1f} %')
pd.DataFrame({'original': original[cambiaron], 'estandarizado': estandarizado[cambiaron]}).head(8)

Nombres no nulos: 48436
Nombres que cambiaron con la estandarización: 6321
Porcentaje: 13.1 %


,original,estandarizado
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg
45,Raisin Cinnamon Bagels 5 count,Raisin Cinnamon Bagels 5 Count
48,Vegetarian Grain Meat Sausages Italian - 4 CT,Vegetarian Grain Meat Sausages Italian - 4 Ct
63,Autumn Vegetable & Turkey Dinner with Lil' Bit...,Autumn Vegetable & Turkey Dinner With Lil' Bit...


**Respuesta:** de los **48 436** nombres no nulos, **6 321 cambiaron** con la estandarización (13,1 %). En los ejemplos
se ve que la mayoría de los cambios vienen de `.str.title()`, que pone en mayúscula palabras como *for*, *with* o *count*;
otros vienen de quitar espacios dobles o invisibles (`Suprema  Frozen` → `Suprema Frozen`). La comparación es fila a fila
porque `products` conserva el mismo orden e índice que el archivo crudo: la limpieza de nombres no elimina filas.

### Ejercicio 5
Crear una Series con el nombre del día de cada pedido, convertirla a categoría **ordenada** de domingo a sábado y
comparar la memoria del texto con la de la categoría.

In [22]:
dias = ['Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']   # 0 = domingo

dia_texto = orders['order_dow'].map(dict(enumerate(dias)))
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=dias, ordered=True), name='dia')

mem_texto = dia_texto.memory_usage(deep=True) / 1e6
mem_categoria = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Memoria como texto:     {mem_texto:.2f} MB')
print(f'Memoria como categoría: {mem_categoria:.2f} MB')
print(f'La categoría ocupa {mem_texto / mem_categoria:.0f} veces menos')
print('¿Ordenada?', dia_categoria.cat.ordered, '| primer día:', dia_categoria.min(), '| último día:', dia_categoria.max())
dia_categoria.value_counts(sort=False)

Memoria como texto:     7.14 MB
Memoria como categoría: 0.48 MB
La categoría ocupa 15 veces menos
¿Ordenada? True | primer día: Domingo | último día: Sábado


dia
Domingo      84090
Lunes        82185
Martes       65833
Miércoles    60897
Jueves       59810
Viernes      63488
Sábado       62649
Name: count, dtype: int64

**Respuesta:** como texto, la Series ocupa **7,14 MB**; como categoría, **0,48 MB**: unas **15 veces menos**
(con pandas 3 y pyarrow; sin pyarrow el texto ocupa aún más y la diferencia crece). La categoría guarda los 7 nombres una sola vez y, para cada
pedido, solo un código entero pequeño; el texto repite el nombre en cada una de las 478 952 filas. Domingo es el día con
más pedidos (84 090) y jueves el que menos (59 810). Al ser **ordenada**, pandas sabe
que domingo va antes que lunes: `value_counts(sort=False)`, `min()`, `max()` y los gráficos respetan el orden de la
semana en lugar del orden alfabético.

### Ejercicio 6
¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen?
¿Se deberían eliminar?

In [23]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]

print('Productos Unknown en el catálogo:', len(ids_unknown))
print('Filas de order_prods con productos Unknown:', len(filas_unknown),
      f'({len(filas_unknown) / len(order_prods) * 100:.2f} % de los artículos)')
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique(),
      f'({filas_unknown["order_id"].nunique() / order_prods["order_id"].nunique() * 100:.2f} % de los pedidos)')

Productos Unknown en el catálogo: 1258
Filas de order_prods con productos Unknown: 9547 (0.21 % de los artículos)


Pedidos en los que aparecen: 8254 (1.83 % de los pedidos)


Los 1 258 productos `'Unknown'` aparecen en **9 547 filas** de `order_prods` (0,21 % de los artículos), repartidas en
**8 254 pedidos** (1,83 % de los pedidos).

**Respuesta:** **no los eliminaría.** Son compras reales: borrarlas dejaría 8 254 pedidos incompletos y alteraría
indicadores como el tamaño del carrito o la tasa de recompra de esos pedidos. Lo adecuado es conservarlos con la etiqueta
`'Unknown'` y excluirlos solo en los análisis que dependen del nombre o de la categoría del producto
(por ejemplo, un ranking de productos más vendidos).

### Ejercicio 7
En una **copia** de `departments`, reemplazar `'missing'` por un nulo real (`pd.NA`) y explicar por qué no conviene
hacerlo en la tabla original antes de agrupar.

In [24]:
departments_na = departments.copy()
departments_na['department'] = departments_na['department'].replace('missing', pd.NA)

print('Nulos en la copia:', departments_na['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())
departments_na.tail(3)

Nulos en la copia: 1
Nulos en la original: 0


,department_id,department
18,19,snacks
19,20,deli
20,21,NaN


En la copia, el departamento 21 queda con un nulo real (1 nulo) y la tabla original sigue con 0.

**Respuesta:** porque `groupby` descarta por defecto las claves nulas (`dropna=True`): si `'missing'` se convierte en
nulo en la tabla original, todos los productos de ese departamento desaparecerían en silencio de cualquier agrupación
por departamento y los totales dejarían de cuadrar.

# Lección 3: Filtrado, ordenamiento y agrupación de datos

Convertiremos las tablas limpias en respuestas: cuándo compran los clientes y con qué frecuencia vuelven.
**Filtrar** elige las filas que importan, **ordenar** las jerarquiza y **agrupar** las resume.
Antes de escribir código, cada pregunta se traduce a la frase "*métrica* por *grupo*": "¿A qué hora se compra más?"
se convierte en "número de pedidos por hora".

### Filtrado con máscaras booleanas

In [25]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


*Esperado:* tres valores booleanos y luego `28817`.

In [26]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


*Esperado:* `45599`. `isin` comprueba si el valor está en una lista, `between` si está en un rango (ambos extremos
incluidos) y `&` exige las dos condiciones (también existen `|` para "o" y `~` para "no").

### Filtrado con `query`

In [27]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


*Esperado:* `1449`: pedidos hechos desde las 10 p. m. por clientes que tardaron 30 días en volver.

### Ordenamiento con `sort_values` y `nlargest`

In [28]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100


        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


*Esperado:* tres clientes con `order_number` 100. En `historial`, el cliente `2` aparece primero, con sus pedidos 1 y 4.

### Agrupación con `groupby`

In [29]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


*Esperado:* `10 40578`: la hora pico es las 10 a. m.

Varias métricas a la vez con **agregación con nombre**:

In [30]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


*Esperado:* el domingo (`0`) tiene más pedidos (84 090) y clientes (57 661) y la mayor espera media (11,8 días);
jueves y viernes (`4` y `5`) tienen la menor espera media (10,5). `'mean'` ignora los nulos, por eso fue correcto no
rellenarlos con 0 en la Lección 2.

### Tabla dinámica con `pivot_table`

In [31]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


*Esperado:* forma `(24, 7)`. La celda más alta es lunes a las 10, con 7 881 pedidos.

### Valores por fila con `transform`

In [32]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


*Esperado:* `231946 30527`: 30 527 clientes con 5 pedidos o más, que suman 231 946 pedidos. `transform` devuelve el
valor del grupo en cada fila original, a diferencia de `agg`, que devuelve una fila por grupo.

## Laboratorio guiado 3: perfil de hábitos de compra

El área de mercadeo quiere diseñar campañas distintas según qué tan seguido compra cada cliente, y necesita saber
**a qué hora compra cada tipo de cliente**. Plan: contar los pedidos de cada cliente, clasificarlos en tres segmentos,
clasificar los pedidos en franjas horarias y cruzar segmentos y franjas.

### Paso 1: pedidos por cliente

In [33]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

*Esperado:* `157437` clientes, con una mediana de 2 pedidos y un máximo de 28.

### Paso 2: segmentar clientes con `pd.cut`

Segmentos: **ocasional** (1 pedido), **regular** (2 a 4) y **frecuente** (5 o más).

In [34]:
import numpy as np

segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


*Esperado:* ocasional 55 357 (35,2 %), regular 71 553 (45,4 %) y frecuente 30 527 (19,4 %).

### Paso 3: llevar el segmento a cada pedido con `map`

In [35]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


`map` busca cada `user_id` en el índice de `segmento_cliente` y devuelve su valor, como un BUSCARV de Excel.

### Paso 4: resumen por segmento

In [36]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


*Esperado:*

| Segmento | Clientes | Pedidos | Espera mediana (días) |
| --- | --- | --- | --- |
| ocasional | 55 357 | 55 357 | 16 |
| regular | 71 553 | 191 649 | 11 |
| frecuente | 30 527 | 231 946 | 6 |

**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231 946 de 478 952) y
vuelve cada 6 días. Los ocasionales tienen espera porque su único pedido en estos datos no es necesariamente el primero
que hicieron en la tienda.

### Paso 5: franjas horarias y tabla cruzada

In [37]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


*Esperado:*

| Segmento | Madrugada | Mañana | Tarde | Noche |
| --- | --- | --- | --- | --- |
| ocasional | 2,1 | 30,0 | 49,4 | 18,5 |
| regular | 1,9 | 31,6 | 48,4 | 18,1 |
| frecuente | 1,7 | 35,2 | 45,9 | 17,2 |

**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana
(35,2 % frente a 30,0 % de los ocasionales). Para mercadeo, una campaña de fidelización debería llegar por la mañana.

### Paso 6: las combinaciones de día y franja más activas

In [38]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

*Esperado:* domingo en la tarde (42 340 pedidos), lunes en la tarde (37 432) y martes en la tarde (30 751).

### Paso 7: dejar `orders` como estaba

In [39]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


*Esperado:* `(478952, 6)`.

**Conclusiones del laboratorio**

- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle, cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

## Reto 3

Cada punto se escribe primero como "métrica por grupo" o como condición, y después en código.

### Ejercicio 1
¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

*Condición + métrica por grupo:* pedidos con `order_dow == 0` → número de pedidos por hora → las 3 mayores.

In [40]:
domingos = orders[orders['order_dow'] == 0]
top_horas_domingo = domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)
top_horas_domingo

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64

**Respuesta:** los domingos, las horas con más pedidos son las **15 h (7 585)**, las **14 h (7 533)** y las
**13 h (7 483)**. A diferencia del total de la semana, cuyo pico es a las 10 a. m., el domingo la compra se concentra
a primera hora de la tarde.

### Ejercicio 2
¿A qué hora es mayor la espera media entre pedidos y a qué hora es menor?

*Métrica por grupo:* media de `days_since_prior_order` por hora.

In [41]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print('Mayor espera media:', espera_por_hora.idxmax(), 'h ->', round(espera_por_hora.max(), 2), 'días')
print('Menor espera media:', espera_por_hora.idxmin(), 'h ->', round(espera_por_hora.min(), 2), 'días')
espera_por_hora.round(2).sort_values()

Mayor espera media: 0 h -> 12.52 días
Menor espera media: 22 h -> 10.62 días


order_hour_of_day
22    10.62
6     10.63
21    10.64
8     10.68
20    10.69
7      10.7
9     10.75
23    10.96
19     11.0
10    11.06
15    11.19
11     11.2
17    11.21
14    11.22
16    11.23
13     11.3
18     11.3
12    11.31
5     11.58
3     11.72
2     12.12
1     12.18
4     12.19
0     12.52
Name: days_since_prior_order, dtype: Float64

**Respuesta:** la mayor espera media es a las **0 h (12,52 días)** y la menor a las **22 h (10,62 días)**, casi empatada
con las 6 h (10,63). Quienes compran de madrugada tardan más en volver; quienes compran temprano en la mañana o en la
noche vuelven antes.

### Ejercicio 3
¿Cuántos clientes tienen 10 pedidos o más?

*Métrica por grupo + condición:* número de pedidos por cliente → clientes con 10 o más.

In [42]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
clientes_10 = (pedidos_por_cliente >= 10).sum()
print('Clientes con 10 pedidos o más:', clientes_10,
      f'({clientes_10 / len(pedidos_por_cliente) * 100:.1f} % de {len(pedidos_por_cliente)} clientes)')

Clientes con 10 pedidos o más: 6327 (4.0 % de 157437 clientes)


**Respuesta:** **6 327 clientes** tienen 10 pedidos o más en estos datos, un 4,0 % de los 157 437 clientes.

### Ejercicio 4
¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

*Condición:* `days_since_prior_order == 7`.

In [43]:
a_los_7 = (orders['days_since_prior_order'] == 7).sum()
print('Pedidos a los 7 días exactos:', a_los_7)
print(f'Porcentaje del total de pedidos: {a_los_7 / len(orders) * 100:.2f} %')
print(f'Porcentaje de los pedidos con pedido anterior: '
      f'{a_los_7 / orders["days_since_prior_order"].notna().sum() * 100:.2f} %')

Pedidos a los 7 días exactos: 44577
Porcentaje del total de pedidos: 9.31 %
Porcentaje de los pedidos con pedido anterior: 9.90 %


**Respuesta:** **44 577 pedidos** se hicieron exactamente una semana después del anterior: el **9,31 %** del total
(9,90 % si solo se cuentan los pedidos que tienen un pedido anterior). Es un pico claro que refleja la compra semanal.

### Ejercicio 5
Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

*Métrica por grupo:* número de productos por pedido.

In [44]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum(), 'de', len(productos_por_pedido))
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420 de 450046
Pedido más grande: 61355 con 127 productos


**Respuesta:** **49 420 pedidos** tienen 20 productos o más (de 450 046 pedidos con detalle). El pedido más grande es el
**61355**, con **127 productos**; es uno de los 70 pedidos con nulos en `add_to_cart_order` del Reto 2.

### Ejercicio 6
Usando `order_prods`: tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito.

*Condición + métrica por grupo:* posiciones 1 a 5 → media de `reordered` por posición.

In [45]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta:** la tasa de recompra **baja a medida que avanza la posición**: 0,679 en la posición 1, 0,677 en la 2,
0,659 en la 3, 0,638 en la 4 y 0,617 en la 5. Los clientes agregan primero los productos que ya compran habitualmente
y después los productos nuevos.

### Ejercicio 7
Comparar la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto.

*Métrica por grupo:* mediana de espera por "es fin de semana".

In [46]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
orders.groupby(es_fin_de_semana)['days_since_prior_order'].median().rename(index={True: 'fin de semana', False: 'resto de la semana'})

order_dow
resto de la semana    7.0
fin de semana         8.0
Name: days_since_prior_order, dtype: Float64

**Respuesta:** la mediana de espera es de **8 días** en los pedidos de fin de semana y de **7 días** en el resto de la
semana. Quienes compran el fin de semana tardan un día más en volver, lo que coincide con la mayor espera media del
domingo (11,8) y del sábado (11,4) en el resumen por día.